# Nested Gaussian rings: the memo example

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/simulation/nested_gaussian_rings_example.ipynb)

The memo *Analytic visibility model for Gaussian-broadened rings* illustrates its ring primitive with
three concentric rings, two sharing a geometry and a third deliberately misaligned, drawn in units of
milliarcseconds.  This notebook recreates that figure with the simulator's own component functions,
``gaussian_ring_image`` (the brightness on the sky) and ``gaussian_ring_uv_response`` (its
visibility), and checks both against the memo's formulas.

A Gaussian-broadened ring of radius $R$ and total flux $F$ is a thin ring convolved with a circular
Gaussian of width $\sigma$, inclined by $i$ ($q = \cos i$) and rotated to position angle $\psi$ (east of
north).  Its brightness in the deprojected radius $r$ is

$$ I(r) = \frac{F}{2\pi\sigma^2 q}\,\exp\!\Big[-\frac{r^2 + R^2}{2\sigma^2}\Big]\,I_0\!\Big(\frac{rR}{\sigma^2}\Big), $$

and its tangent-plane visibility is $F\,J_0(2\pi R\rho_{\rm ell})\exp(-2\pi^2\sigma^2\rho_{\rm ell}^2)$.  The
simulator parametrises the broadening by its FWHM, $2\sqrt{2\ln 2}\,\sigma$, and all angles in radians.

---
## API

In [ ]:
from astroviper.processing_functions.simulation import (
    gaussian_ring_image,
    gaussian_ring_uv_response,
)

gaussian_ring_image?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
# Set True for interactive (zoom / pan) plots via the ipympl widget backend
# (``pip install ipympl``).  Keep False for the automated notebook tests, which
# execute headless.
INTERACTIVE_PLOTS = False

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_PLOTS else "inline")

MAS_TO_RAD = np.deg2rad(1.0 / 3_600_000.0)
SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))

## The three rings of the memo

Rings 1 and 2 share the inclination and position angle; ring 3 is misaligned.  The memo gives the
Gaussian width $\sigma$; the simulator's shape vector is ``[radius, fwhm, inclination, position_angle]``.

In [ ]:
rings_mas = pd.DataFrame(
    [
        {
            "flux [Jy]": 0.003,
            "radius [mas]": 20.0,
            "sigma [mas]": 3.0,
            "inclination [deg]": 40.0,
            "pa [deg]": 35.0,
        },
        {
            "flux [Jy]": 0.010,
            "radius [mas]": 40.0,
            "sigma [mas]": 4.5,
            "inclination [deg]": 40.0,
            "pa [deg]": 35.0,
        },
        {
            "flux [Jy]": 0.020,
            "radius [mas]": 65.0,
            "sigma [mas]": 6.0,
            "inclination [deg]": 55.0,
            "pa [deg]": 70.0,
        },
    ],
    index=["ring 1", "ring 2", "ring 3"],
)
fluxes = rings_mas["flux [Jy]"].to_numpy()
shapes = np.stack(
    [
        rings_mas["radius [mas]"].to_numpy() * MAS_TO_RAD,
        rings_mas["sigma [mas]"].to_numpy() * MAS_TO_RAD * SIGMA_TO_FWHM,
        np.deg2rad(rings_mas["inclination [deg]"].to_numpy()),
        np.deg2rad(rings_mas["pa [deg]"].to_numpy()),
    ],
    axis=1,
)  # [ring, (radius, fwhm, inclination, pa)] in radians
rings_mas

## The image

The memo's grid: 700 x 700 pixels over $\pm 100$ mas, the brightness converted from Jy per steradian
to mJy per square milliarcsecond, and a few contours that make the misaligned outer ring obvious.

In [ ]:
extent_mas = 100.0
n = 700
coord_mas = np.linspace(-extent_mas, extent_mas, n)
x_mas, y_mas = np.meshgrid(
    coord_mas, coord_mas
)  # x = delta RA (east), y = delta Dec (north)
x = x_mas * MAS_TO_RAD
y = y_mas * MAS_TO_RAD

image = np.zeros_like(x)
for flux, shape in zip(fluxes, shapes, strict=True):
    image += flux * gaussian_ring_image(x, y, *shape)
image_plot = image * MAS_TO_RAD**2 * 1e3  # mJy / mas^2

fig, ax = plt.subplots(figsize=(6.6, 6.0))
im = ax.imshow(
    image_plot,
    origin="lower",
    extent=[-extent_mas, extent_mas, -extent_mas, extent_mas],
    interpolation="nearest",
)
positive = image_plot[image_plot > 0]
levels = np.geomspace(positive.max() * 0.08, positive.max() * 0.65, 5)
ax.contour(x_mas, y_mas, image_plot, levels=levels, linewidths=0.8)
ax.set_xlabel(r"$\Delta$RA [mas]")
ax.set_ylabel(r"$\Delta$Dec [mas]")
ax.set_title("Three analytic Gaussian-broadened rings")
ax.set_aspect("equal")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r"Surface brightness [mJy mas$^{-2}$]")
fig.tight_layout()
plt.show()

## Checks against the memo

The image integrates to the total flux of the rings, and the simulator's functions agree with the
memo's formulas (evaluated here as written in the memo) to rounding, both on the sky and in the
visibility domain.

In [ ]:
from scipy.special import i0e, j0

pixel_area = (coord_mas[1] - coord_mas[0]) ** 2 * MAS_TO_RAD**2
print(
    f"integrated flux of the image: {image.sum() * pixel_area * 1e3:.4f} mJy (input {fluxes.sum() * 1e3:.1f} mJy)"
)


def memo_brightness(x, y, flux, radius, sigma, inc, pa):
    """Equation (6) of the memo, projected: the memo script's ring_brightness."""
    q = np.cos(inc)
    x_major = x * np.sin(pa) + y * np.cos(pa)
    x_minor = x * np.cos(pa) - y * np.sin(pa)
    r = np.sqrt(x_major**2 + (x_minor / q) ** 2)
    radial = np.exp(-0.5 * ((r - radius) / sigma) ** 2) * i0e(r * radius / sigma**2)
    return flux / (2.0 * np.pi * sigma**2 * q) * radial


def memo_visibility(u, v, flux, radius, sigma, inc, pa):
    """Equation (11) of the memo (tangent plane): the memo script's ring_visibility."""
    q = np.cos(inc)
    u_major = u * np.sin(pa) + v * np.cos(pa)
    u_minor = u * np.cos(pa) - v * np.sin(pa)
    rho = np.sqrt(u_major**2 + (q * u_minor) ** 2)
    return (
        flux
        * j0(2.0 * np.pi * radius * rho)
        * np.exp(-2.0 * np.pi**2 * sigma**2 * rho**2)
    )


memo_image = np.zeros_like(x)
for flux, (radius, fwhm, inc, pa) in zip(fluxes, shapes, strict=True):
    memo_image += memo_brightness(x, y, flux, radius, fwhm / SIGMA_TO_FWHM, inc, pa)
print(
    f"max |simulator image - memo image| = {np.abs(image - memo_image).max() * MAS_TO_RAD**2 * 1e3:.2e} mJy/mas^2"
)

rng = np.random.default_rng(1)
u = rng.uniform(
    -3e9, 3e9, 2000
)  # up to 20 mega-wavelengths, where mas rings are resolved
v = rng.uniform(-2e7, 2e7, 2000)
simulator_v = sum(
    flux * gaussian_ring_uv_response(u, v, *shape)
    for flux, shape in zip(fluxes, shapes, strict=True)
)
memo_v = sum(
    memo_visibility(u, v, flux, radius, fwhm / SIGMA_TO_FWHM, inc, pa)
    for flux, (radius, fwhm, inc, pa) in zip(fluxes, shapes, strict=True)
)
print(
    f"max |simulator visibility - memo visibility| = {np.abs(simulator_v - memo_v).max():.2e} Jy"
)
print(
    f"V(0, 0) = {sum(flux * gaussian_ring_uv_response(0.0, 0.0, *shape) for flux, shape in zip(fluxes, shapes, strict=True)) * 1e3:.4f} mJy"
)
assert np.abs(image - memo_image).max() < 1e-12 * image.max()
assert np.abs(simulator_v - memo_v).max() < 1e-12

## Visibility amplitude against baseline

The three rings have different radii, so their Bessel oscillations beat; the misaligned ring makes the
amplitude depend on the baseline orientation, shown here along the position angles of the two ring
geometries.  Structure of a few tens of milliarcseconds is resolved beyond a few mega-wavelengths:
tens of kilometres at 7 mm, i.e. VLA A-configuration to VLBI baselines.

In [ ]:
baseline = np.linspace(0, 2e7, 1500)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for pa_deg, style in [(35.0, "-"), (70.0, "--")]:
    pa = np.deg2rad(pa_deg)
    total = sum(
        flux
        * gaussian_ring_uv_response(
            baseline * np.sin(pa), baseline * np.cos(pa), *shape
        )
        for flux, shape in zip(fluxes, shapes, strict=True)
    )
    ax.plot(
        baseline / 1e6,
        np.abs(total) * 1e3,
        style,
        label=f"baseline along pa = {pa_deg:.0f} deg",
    )
ax.set_xlabel("baseline length [M$\\lambda$]")
ax.set_ylabel("|V| [mJy]")
ax.set_title("Three nested Gaussian rings")
ax.grid(alpha=0.3)
ax.legend()
plt.show()